# Clustered Curve Beta CDF Model

This notebook consumes the CSV produced by `custpaydetails_clustered_cumulative_curves.sql`, fits Beta CDF expected cumulative burn curves, evaluates held-out items, and compares the Beta CDF approach to a pure linear spend model.

## Input and Parameterization

| Metric | Value |
| --- | --- |
| Input CSV | clustered_data_input_allcustomers.csv |
| Cluster curve rows | 9,303 |
| Items | 1,003 |
| Train items | 814 |
| Test items | 189 |
| Global Beta alpha | 0.533320 |
| Global Beta beta | 0.806129 |
| Global train MSE | 0.04183106 |
| Proxy label CSV | clustered_curve_proxy_labels_allcustomers.csv |
| Proxy-labeled held-out rows | 1,761 |

In [ ]:
import csv
from pathlib import Path

path = Path('clustered_data_input.csv')
if not path.exists():
    path = Path('custpaydetails_clustered_cumulative_curves.csv')
if not path.exists():
    path = Path('ci_item_clustered_cumulative_curves.csv')
with path.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
print(path)
print(len(rows))
print(reader.fieldnames)

## Linear Spend Baseline

The pure linear model assumes cumulative spend should equal elapsed time:

```text
expected_cumulative_pct_linear = elapsed_pct
position_delta_linear = actual_cumulative_pct - elapsed_pct
```

This is a useful baseline because it is transparent and easy to explain. It is also too rigid: many items are naturally front-loaded or back-loaded, so a linear curve can systematically flag normal burn shapes as risk.

## Held-Out Model Performance

Errors are cumulative-spend-percent errors. MAE `0.15` means an average absolute error of about 15 percentage points of final item spend.

| Model | MAE | RMSE | Median AE | P90 AE | Bias |
| --- | --- | --- | --- | --- | --- |
| Duration-bucket Beta CDF | 0.1454 | 0.1970 | 0.1121 | 0.3284 | 0.0212 |
| Cluster-count-bucket Beta CDF | 0.1481 | 0.1999 | 0.1179 | 0.3414 | 0.0154 |
| Pooled empirical median curve | 0.1488 | 0.2067 | 0.1099 | 0.3430 | 0.0121 |
| Global Beta CDF | 0.1498 | 0.2033 | 0.1166 | 0.3410 | 0.0169 |
| Linear cumulative spend | 0.1640 | 0.2294 | 0.1212 | 0.3904 | -0.0821 |

## Beta CDF vs Linear: Improvement Summary

| Comparison | Value |
| --- | --- |
| MAE improvement vs linear | 11.38% |
| RMSE improvement vs linear | 14.10% |
| P90 absolute-error improvement vs linear | 15.86% |
| Linear bias | -0.0821 |
| Duration-bucket Beta bias | 0.0212 |

## Error Comparison Plot



## Bucketed Beta CDF Parameters

### Duration Buckets

| Duration bucket | alpha | beta |
| --- | --- | --- |
| 181-365d | 0.557407 | 0.904785 |
| 366-730d | 0.567542 | 0.932235 |
| <=180d | 0.800024 | 0.745257 |
| >730d | 0.606764 | 1.125182 |

### Cluster-Count Buckets

| Cluster-count bucket | alpha | beta |
| --- | --- | --- |
| 13+ clusters | 0.740344 | 1.174418 |
| 4-6 clusters | 0.239743 | 0.319744 |
| 7-12 clusters | 0.482847 | 0.757897 |
| <=3 clusters | 0.142027 | 0.162297 |

## Curve Performance Plot



## Risk Signal Framing

For active items, the same fitted curve can produce budget-overrun and time-overrun risk signals. These are not final labels without a real authorized budget and schedule; they are position signals.

```text
expected_pct = model(elapsed_pct)
position_delta = actual_cumulative_pct - expected_pct

if position_delta > threshold:
    spending too quickly / budget-overrun risk

if position_delta < -threshold:
    spending too slowly / time-overrun risk
```

For budget overrun projection once a budget is available:

```text
projected_final_spend = actual_cumulative_spend / expected_pct
projected_overrun = projected_final_spend - authorized_budget
```

## Beta vs Linear Risk Signals on Held-Out Data

The table shows how many clustered observations would be flagged by each approach at several position-delta thresholds. `LinearOnly` rows are especially important: these are cases the linear model flags but the duration-bucket Beta curve treats as normal for the observed burn shape.

| Threshold | Signal | Beta count | Linear count | Both | Linear only | Beta only |
| --- | --- | --- | --- | --- | --- | --- |
| 0.10 | spending too quickly / budget-overrun risk | 392 | 714 | 381 | 333 | 11 |
| 0.10 | spending too slowly / time-overrun risk | 558 | 271 | 262 | 9 | 296 |
| 0.15 | spending too quickly / budget-overrun risk | 277 | 562 | 266 | 296 | 11 |
| 0.15 | spending too slowly / time-overrun risk | 448 | 182 | 179 | 3 | 269 |
| 0.25 | spending too quickly / budget-overrun risk | 146 | 337 | 143 | 194 | 3 |
| 0.25 | spending too slowly / time-overrun risk | 214 | 79 | 73 | 6 | 141 |

## Proxy ROC/AUC Setup

True ROC/AUC requires true binary outcome labels. This notebook now consumes retrospective proxy labels from `clustered_curve_proxy_labels.csv`, which is generated by the separate polynomial proxy-label notebook. The Beta CDF and linear models do not create those labels; they only score against them.

These ROC curves compare how well Beta and linear position scores recover the external proxy labels. They are useful for model behavior comparison, but they are not a substitute for true budget/schedule outcome labels.

## Proxy ROC/AUC Summary

| Signal | Model | AUC | Positive labels | Negative labels |
| --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.9901 | 310 | 1,451 |
| fast_spend_proxy | Linear cumulative spend | 0.9870 | 310 | 1,451 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.9825 | 426 | 1,335 |
| slow_spend_proxy | Linear cumulative spend | 0.9756 | 426 | 1,335 |

## Threshold Sweep for Proxy Labels

| Signal | Model | Threshold | Flagged | TP | FP | Precision | Recall/TPR | FPR |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.05 | 518 | 310 | 208 | 0.598 | 1.000 | 0.143 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.10 | 392 | 304 | 88 | 0.776 | 0.981 | 0.061 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.15 | 277 | 251 | 26 | 0.906 | 0.810 | 0.018 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.20 | 202 | 193 | 9 | 0.955 | 0.623 | 0.006 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.25 | 146 | 144 | 2 | 0.986 | 0.465 | 0.001 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.30 | 106 | 106 | 0 | 1.000 | 0.342 | 0.000 |
| fast_spend_proxy | Duration-bucket Beta CDF | 0.40 | 52 | 52 | 0 | 1.000 | 0.168 | 0.000 |
| fast_spend_proxy | Linear cumulative spend | 0.05 | 863 | 310 | 553 | 0.359 | 1.000 | 0.381 |
| fast_spend_proxy | Linear cumulative spend | 0.10 | 714 | 310 | 404 | 0.434 | 1.000 | 0.278 |
| fast_spend_proxy | Linear cumulative spend | 0.15 | 562 | 310 | 252 | 0.552 | 1.000 | 0.174 |
| fast_spend_proxy | Linear cumulative spend | 0.20 | 425 | 302 | 123 | 0.711 | 0.974 | 0.085 |
| fast_spend_proxy | Linear cumulative spend | 0.25 | 337 | 270 | 67 | 0.801 | 0.871 | 0.046 |
| fast_spend_proxy | Linear cumulative spend | 0.30 | 259 | 237 | 22 | 0.915 | 0.765 | 0.015 |
| fast_spend_proxy | Linear cumulative spend | 0.40 | 147 | 147 | 0 | 1.000 | 0.474 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.05 | 715 | 416 | 299 | 0.582 | 0.977 | 0.224 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.10 | 558 | 401 | 157 | 0.719 | 0.941 | 0.118 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.15 | 448 | 389 | 59 | 0.868 | 0.913 | 0.044 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.20 | 329 | 325 | 4 | 0.988 | 0.763 | 0.003 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.25 | 214 | 214 | 0 | 1.000 | 0.502 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.30 | 117 | 117 | 0 | 1.000 | 0.275 | 0.000 |
| slow_spend_proxy | Duration-bucket Beta CDF | 0.40 | 37 | 37 | 0 | 1.000 | 0.087 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.05 | 386 | 333 | 53 | 0.863 | 0.782 | 0.040 |
| slow_spend_proxy | Linear cumulative spend | 0.10 | 271 | 260 | 11 | 0.959 | 0.610 | 0.008 |
| slow_spend_proxy | Linear cumulative spend | 0.15 | 182 | 182 | 0 | 1.000 | 0.427 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.20 | 123 | 123 | 0 | 1.000 | 0.289 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.25 | 79 | 79 | 0 | 1.000 | 0.185 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.30 | 55 | 55 | 0 | 1.000 | 0.129 | 0.000 |
| slow_spend_proxy | Linear cumulative spend | 0.40 | 18 | 18 | 0 | 1.000 | 0.042 | 0.000 |

## Proxy ROC Curves



## Residual Distribution Plot

Residual is `actual cumulative pct - expected cumulative pct`. Positive residuals indicate spending ahead of expected position; negative residuals indicate spending behind expected position.



## Recommendations

Use the duration-bucket Beta CDF as the first expected-position model and keep the linear model as a transparent benchmark. For production alerting:

- Use Beta CDF `position_delta` for primary spend-too-fast and spend-too-slow signals.
- Show the linear delta as a secondary reference because users understand it.
- Alert only when position delta exceeds a threshold and remains elevated across updates.
- For budget-overrun detection, combine position delta with authorized budget and projected final spend.
- For time-overrun detection, combine slow-spend position delta with schedule metadata; slow spending can mean delay, scope removal, or inactive work.